# 08 — Massimizzare DHSLP

DHSLP era debole (dep 0.24) e **overfittava**. Diagnosi: il collo di bottiglia **non** è l'ipergrafo
né le finestre, ma l'**encoder dei nodi** — un semplice `Linear` sui campioni grezzi non impara filtri
di frequenza, mentre EEGNet/Shallow funzionano proprio grazie alle **convoluzioni temporali**.

Leve provate (in ordine di importanza):
1. **`node_encoder='conv'`** — encoder temporale convoluzionale per nodo (come EEGNet/Shallow). *La leva chiave.*
2. **ipergrafo `pruned`** da connettività PCC/PLV (struttura fissa → meno overfitting) e **`hybrid`**.
3. **regolarizzazione** (dropout, weight decay, label smoothing).

> ⚠️ **Aspettativa onesta**: col conv encoder DHSLP dovrebbe salire, ma il **mean-pool sui nodi** è un
> aggregatore spaziale più debole della conv spaziale di EEGNet → realistico ~0.35–0.45, forse non 0.55.
> E la connettività resta cieca alla parola (word NMI≈0). Obiettivo: DHSLP il più forte possibile + verdetto onesto.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import track3_config as C, track3_train as T
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Sweep rapido su 5 soggetti (trova la config migliore)
Ogni config su S01–S05 (subject-dependent, 100 epoche). Guarda **test** e **train** (gap = overfitting).

In [ ]:
BASE_TK = dict(epochs=100, patience=20, lr=1e-3, batch_size=32)
CONFIGS = [
    # --- baseline (encoder Linear) ---
    ('linear_learned',      dict(node_encoder='linear', hyperedge_mode='learned'), {}),
    # --- la leva chiave: encoder CONV ---
    ('conv_learned',        dict(node_encoder='conv', hyperedge_mode='learned'), {}),
    ('conv_learned+reg',    dict(node_encoder='conv', hyperedge_mode='learned', dropout=0.5), dict(label_smoothing=0.1)),
    ('conv_pruned_pcc',     dict(node_encoder='conv', hyperedge_mode='pruned', metric='pcc', k_neighbors=8), {}),
    ('conv_pruned_plv',     dict(node_encoder='conv', hyperedge_mode='pruned', metric='plv', k_neighbors=8), {}),
    ('conv_hybrid_pcc',     dict(node_encoder='conv', hyperedge_mode='hybrid', metric='pcc', k_neighbors=8), {}),
    # --- pruned/hybrid con encoder Linear (per isolare l'effetto dell'encoder) ---
    ('linear_pruned_pcc',   dict(node_encoder='linear', hyperedge_mode='pruned', metric='pcc', k_neighbors=8), {}),
    ('linear_hybrid_pcc',   dict(node_encoder='linear', hyperedge_mode='hybrid', metric='pcc', k_neighbors=8), {}),
]
rows = []
for name, mk, tk in CONFIGS:
    df, _ = T.run_subject_dependent('dhslp', subjects=[1,2,3,4,5], pp_kwargs=C.PP_MINIMAL,
                                    model_kwargs=mk, train_kwargs={**BASE_TK, **tk}, verbose=False)
    rows.append({'config': name, 'test': df.test_acc.mean(), 'train': df.train_acc.mean(),
                 'gap': df.train_acc.mean()-df.test_acc.mean()})
    print(f"{name:20s} test={rows[-1]['test']:.3f}  train={rows[-1]['train']:.3f}  gap={rows[-1]['gap']:+.3f}")
sweep = pd.DataFrame(rows).set_index('config').sort_values('test', ascending=False)
print('\nchance =', C.CHANCE_LEVEL); sweep.round(3)

## §2 — Config migliore su TUTTI i 15 soggetti (subject-dependent)
Prendi la config in cima e valutala per intero (più epoche).

In [ ]:
# imposta qui la config vincente dallo sweep (esempio: pruned_pcc_k8)
BEST_MK = dict(hyperedge_mode='pruned', metric='pcc', k_neighbors=8)   # <-- adatta al vincitore §1
BEST_TK = dict(epochs=200, patience=30, lr=1e-3, batch_size=32, label_smoothing=0.1)
df_best, res_best = T.run_subject_dependent('dhslp', pp_kwargs=C.PP_MINIMAL,
                                            model_kwargs=BEST_MK, train_kwargs=BEST_TK)
T.save_metrics(df_best, 'dhslp_maxed')
print(f"DHSLP maxed subject-dependent: {df_best.test_acc.mean():.3f} ± {df_best.test_acc.std():.3f}")
T.plot_per_subject(df_best, 'dhslp_maxed'); plt.show()

## §3 — Config migliore su mixed e independent

In [ ]:
df_mix, _ = T.run_subject_mixed('dhslp', pp_kwargs=C.PP_MINIMAL, model_kwargs=BEST_MK, train_kwargs=BEST_TK)
df_ind, _ = T.run_subject_independent('dhslp', mode='holdout', pp_kwargs=C.PP_MINIMAL, model_kwargs=BEST_MK, train_kwargs=BEST_TK)
print('mixed:', df_mix.loc['ALL','test_acc'], '| independent(holdout):', df_ind.iloc[0]['test_acc'])

## §4 — Confronto con il tetto delle ConvNet e conclusioni

In [ ]:
compare = pd.Series({
    'DHSLP originale (dep)': 0.241,
    'DHSLP maxed (dep)':     df_best.test_acc.mean(),
    'EEGNet (dep)':          0.555,
    'ShallowNet (dep)':      0.575,
}).round(3)
print('Subject-dependent, chance', C.CHANCE_LEVEL)
print(compare.to_string())
delta = df_best.test_acc.mean() - 0.241
print(f'\nMiglioramento DHSLP: {delta:+.3f}')
if df_best.test_acc.mean() > 0.45:
    print('=> DHSLP ora competitivo con le ConvNet: risultato forte.')
elif delta > 0.05:
    print('=> DHSLP migliorato ma ancora sotto le ConvNet: il pruned/reg aiuta, non ribalta.')
else:
    print('=> DHSLP resta a chance: conferma che la connettivita non porta info sulla parola.')

### Nota
Qualunque sia l'esito, è un risultato di tesi onesto:
- Se DHSLP maxed sale verso le ConvNet → il pruned + regolarizzazione funziona, DHSLP diventa competitivo.
- Se resta sotto → conferma quantitativa che **la struttura di connettività non aggiunge informazione
  sulla parola** (word NMI≈0), coerente con tutto il resto dell'analisi. In tesi: 'abbiamo massimizzato
  DHSLP con ipergrafi pruned e regolarizzazione; il tetto resta sotto le ConvNet end-to-end sul raw'.